In [1]:

from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
from pandera import Column, Check, DataFrameSchema

API_URL = "https://api.worldbank.org/v2/country/all/indicator/SP.RUR.TOTL.ZS"

# ==========================
# Extracción
# ==========================
@task
def extract_rural_population(year_from: int = 2000, year_to: int = 2023, per_page: int = 1000):
    all_data = []
    page = 1
    total_pages = None

    while True:
        url = f"{API_URL}?format=json&date={year_from}:{year_to}&per_page={per_page}&page={page}"
        response = requests.get(url)
        response.raise_for_status()
        json_data = response.json()

        if total_pages is None:
            total_pages = json_data[0]["pages"]

        data_page = json_data[1]
        all_data.extend(data_page)

        print(f"🔄 Descargando página {page} de {total_pages} ({len(all_data)} filas acumuladas)")

        if page >= total_pages:
            break
        page += 1

    return all_data

# ==========================
# Transformación
# ==========================
@task
def transform_rural_population(data):
    df = pd.DataFrame(data)

    # Seleccionar columnas relevantes
    df_clean = df[["countryiso3code", "country", "date", "value"]].copy()

    # Extraer nombre del país
    df_clean["country_name"] = df_clean["country"].apply(
        lambda x: x.get("value") if isinstance(x, dict) else None
    )

    # Renombrar columnas
    df_clean = df_clean.rename(columns={
        "countryiso3code": "id_country",
        "date": "year",
        "value": "rural_population_pct"
    })[["id_country", "country_name", "year", "rural_population_pct"]]

    # Convertir año a int
    df_clean["year"] = df_clean["year"].astype(int)

    #  Filtrar: solo países con id_country de 3 letras
    df_clean = df_clean[df_clean["id_country"].str.len() == 3]

    return df_clean

# ==========================
# Validación
# ==========================
rural_schema = DataFrameSchema({
    "id_country": Column(str, [
        Check.str_length(3, 3)
    ]),
    "country_name": Column(str, nullable=True),
    "year": Column(int, [
        Check.ge(1900),
        Check.le(2100)
    ]),
    "rural_population_pct": Column(float, nullable=True, checks=[
        Check.ge(0),
        Check.le(100)
    ]),
})

@task
def validate_rural_population(df: pd.DataFrame) -> pd.DataFrame:
    return rural_schema.validate(df)

# ==========================
# Carga
# ==========================
@task
def load_rural_population(df: pd.DataFrame):
    file_path = Path("../stage/country_rural_population.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"💾 Guardado {len(df)} filas en {file_path}")

# ==========================
# Flow
# ==========================
@flow(name="etl_rural_population_flow")
def etl_rural_population(year_from: int = 2000, year_to: int = 2023):
    data = extract_rural_population(year_from, year_to)
    df = transform_rural_population(data)
    df = validate_rural_population(df)
    load_rural_population(df)

if __name__ == "__main__":
    etl_rural_population(year_from=2000, year_to=2025)


C:\Users\Sebastian\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandera\_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


05:58:30.329 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8075
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

05:58:34.629 | INFO    | Flow run 'diligent-boar' - Beginning flow run 'diligent-boar' for flow 'etl_rural_population_flow'

🔄 Descargando página 1 de 7 (1000 filas acumuladas)
🔄 Descargando página 2 de 7 (2000 filas acumuladas)
🔄 Descargando página 3 de 7 (3000 filas acumuladas)
🔄 Descargando página 4 de 7 (4000 filas acumuladas)
🔄 Descargando página 5 de 7 (5000 filas acumuladas)
🔄 Descargando página 6 de 7 (6000 filas acumuladas)
🔄 Descargando página 7 de 7 (6384 filas acumuladas)


05:58:54.380 | INFO    | Task run 'extract_rural_population-688' - Finished in state Completed()

05:58:55.439 | INFO    | Task run 'transform_rural_population-39b' - Finished in state Completed()

05:58:55.676 | INFO    | Task run 'validate_rural_population-085' - Finished in state Completed()

💾 Guardado 6264 filas en ..\stage\country_rural_population.csv


05:58:55.903 | INFO    | Task run 'load_rural_population-aa4' - Finished in state Completed()

05:58:55.918 | INFO    | Flow run 'diligent-boar' - Finished in state Completed()